# CommonsenseQA — CoT Baseline Evaluation
## Chain-of-Thought vs No-Guidance Baseline

**Purpose:** Answers the reviewer question: *does the fine-tuned guide add value beyond simply prompting the solver to think step by step?*

Runs TWO conditions on the same 900 CommonsenseQA questions (seed=42):
- **CoT:** Qwen2.5-1.5B × 5 votes — `Let's think step by step` in system prompt
- **Baseline:** Qwen2.5-1.5B × 5 votes — standard prompt, no guide

| Condition | Compute | Description |
|---|---|---|
| Baseline | 7.5B | No guide, no CoT |
| **CoT (this notebook)** | **7.5B** | No guide, think step by step |
| Guided (original notebook) | 10.5B | Fine-tuned 3B guide + 1.5B solver |

**Same seed=42, same 900 questions — direct 3-way comparison is valid.**

> Known guided result: **75.8%** · Known baseline: **70.7%**

In [ ]:
# CELL 1 -- Install
# !pip install -q transformers==4.44.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

In [ ]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login

login("YOUR_HF_TOKEN_HERE")  # paste your token here
print("Login done")

In [ ]:
# CELL 3 -- Imports + GPU check
import os, json, re, random, time
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/csqa_cot_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")

In [ ]:
# CELL 4 -- Configuration
# CRITICAL: N=900 and seed=42 MUST match the original guided CSQA run.
CONFIG = {
    "solver_model"     : "Qwen/Qwen2.5-1.5B-Instruct",
    "dataset_name"     : "tau/commonsense_qa",
    "dataset_split"    : "validation",
    "max_eval_samples" : 900,   # MUST match original guided run
    "random_seed"      : 42,    # MUST match original guided run
    "n_votes"          : 5,
    "vote_temperature" : 0.4,   # MUST match original guided run
    "max_new_tokens"   : 200,   # MCQ only needs short generation
    "solver_params_B"  : 1.5,
    "results_file"     : f"{OUTPUT_DIR}/results.jsonl",
    "angle1_file"      : f"{OUTPUT_DIR}/angle1_compute.json",
    "angle2_file"      : f"{OUTPUT_DIR}/angle2_consistency.json",
    "angle3_file"      : f"{OUTPUT_DIR}/angle3_calibration.json",
    "checkpoint_file"  : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"       : 50,    # checkpoint every 50 — safe for Colab
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")

In [ ]:
# CELL 5 -- Load CommonsenseQA
# Fields: id, question, question_concept, choices (label + text lists), answerKey

print("Loading CommonsenseQA...")
raw_ds = load_dataset(CONFIG["dataset_name"])
print(f"Validation size: {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample:")
print(f"  Q: {ex['question']}")
print(f"  Choices: {list(zip(ex['choices']['label'], ex['choices']['text']))}")
print(f"  Answer: {ex['answerKey']}")


def normalise_csqa(item):
    q = item["question"].strip()
    choices = list(zip(item["choices"]["label"], item["choices"]["text"]))
    choice_str = "  ".join(f"{lbl}. {txt}" for lbl, txt in choices)
    full_q = f"{q}\nChoices: {choice_str}"
    return {"question": full_q, "answer": item["answerKey"].strip().upper()}


all_data = [normalise_csqa(x) for x in raw_ds[CONFIG["dataset_split"]]]

# Fix seed ONCE -- must match original guided run
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
else:
    test_data = all_data

ans_dist = Counter(d["answer"] for d in test_data)
print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
print("Answer key distribution (should be roughly 20% each):")
for opt in ['A','B','C','D','E']:
    n = ans_dist.get(opt, 0)
    print(f"  {opt}: {n} ({n/len(test_data)*100:.1f}%)")

In [ ]:
# CELL 6 -- Answer extraction for MCQ (A/B/C/D/E)

def extract_mcq_answer(text):
    """Extract single letter A-E from model output. Returns empty string on failure."""
    text = text.strip()
    # 1. Explicit answer statement
    m = re.search(r"(?:the answer is|answer is|answer:|correct answer is)\s*\**([A-E])\**",
                  text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # 2. Therefore/so/thus + letter
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\**([A-E])\**",
                  text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # 3. Last non-empty line is a single letter
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if lines:
        m = re.match(r"^\**([A-E])\**[.)]?$", lines[-1], re.IGNORECASE)
        if m: return m.group(1).upper()
    # 4. Bold letter at end **X**
    m = re.search(r"\*\*([A-E])\*\*\.?\s*$", text)
    if m: return m.group(1).upper()
    # 5. Letter in parentheses at end
    m = re.search(r"\(([A-E])\)\s*$", text)
    if m: return m.group(1).upper()
    # 6. Any A-E at very end of text
    m = re.search(r"\b([A-E])\b[.\s]*$", text)
    if m: return m.group(1).upper()
    return ""


# Self-test
_tests = [
    ("The answer is A", "A"), ("answer: B", "B"),
    ("Therefore, C", "C"), ("\nD", "D"), ("**E**.", "E"),
    ("(A)", "A"), ("no letter here", ""),
]
all_ok = all(extract_mcq_answer(t) == e for t, e in _tests)
print("Extractor:", "ALL PASSED" if all_ok else "FAILURES -- fix before running")
for t, e in _tests:
    got = extract_mcq_answer(t)
    print(f"  {'OK' if got==e else 'FAIL':4}  {t!r:30} -> {got!r}")

In [ ]:
# CELL 7 -- Load solver (Qwen 1.5B only — no guide model)
# T4: 15GB VRAM. 1.5B float16 ~ 3GB. Plenty of headroom.

print(f"Loading: {CONFIG['solver_model']}")
solver_tok = AutoTokenizer.from_pretrained(CONFIG["solver_model"])
if solver_tok.pad_token is None:
    solver_tok.pad_token = solver_tok.eos_token

solver_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["solver_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

vram = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used : {vram:.2f} GB  |  Headroom: {15.0 - vram:.1f} GB")
print("Solver ready")

In [ ]:
# CELL 8 -- Prompts and generation

COT_SYSTEM = (
    "You are a precise reasoning assistant.\n"
    "Let's think step by step.\n"
    "Read the question and all choices carefully.\n"
    "Reason through each option before deciding.\n"
    "Your FINAL line must be exactly: The answer is [letter]"
)

BASELINE_SYSTEM = (
    "You are a precise reasoning assistant.\n"
    "Read the question and all choices carefully.\n"
    "Your FINAL line must be exactly: The answer is [letter]"
)

REFINER_SYSTEM = (
    "You are a careful reasoning assistant.\n"
    "Previous attempts gave conflicting answers. Ignore all of them.\n"
    "Re-read the question and choices completely from scratch.\n"
    "Your FINAL line must be exactly: The answer is [letter]"
)


def run_solver(messages, temperature):
    prompt = solver_tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = solver_tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(solver_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = solver_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=max(temperature, 0.05),
            do_sample=True,
            top_p=0.92, top_k=40,
            pad_token_id=solver_tok.eos_token_id,
            repetition_penalty=1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return solver_tok.decode(new_toks, skip_special_tokens=True).strip()


def generate_cot(question):
    return run_solver(
        [{"role": "system", "content": COT_SYSTEM},
         {"role": "user",   "content": f"Question:\n{question}"}],
        temperature=CONFIG["vote_temperature"],
    )


def generate_baseline(question):
    return run_solver(
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": f"Question:\n{question}"}],
        temperature=CONFIG["vote_temperature"],
    )


def generate_refiner(question, candidates):
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Question:\n{question}\n\n"
        f"Previous attempts gave: {cands}\n"
        "Re-read and decide from scratch:"
    )
    return run_solver(
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        temperature=0.3,
    )


print("Generation functions ready")
print("  CoT      : 'Let's think step by step'")
print("  Baseline : standard prompt")
print("  Temp     :", CONFIG["vote_temperature"])

In [ ]:
# CELL 9 -- Voting logic

def vote_and_decide(answers, question, gt_answer=None):
    valid = [a for a in answers if a and a.strip()]
    if not valid: valid = answers

    vote_counts  = Counter(valid)
    most_common  = vote_counts.most_common()
    top_answer   = most_common[0][0]
    top_count    = most_common[0][1]
    total        = len(valid)
    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used = False; refiner_correct = None

    if is_majority:
        final = top_answer; strategy = "majority"
        conf = round(top_count / total, 4); wasted = total - top_count
    else:
        ref_raw = generate_refiner(question, list(valid))
        ref_ans = extract_mcq_answer(ref_raw)
        refiner_used = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None
        all_v = valid + ([ref_ans] if ref_ans else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top = new_common[0][0]; new_top_c = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]
        final = new_top
        strategy = "coin_flip" if still_tied else "refiner_tiebreak"
        conf = round(new_top_c / len(all_v), 4)
        total = len(all_v)
        correct_votes = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted = total - new_top_c
        vote_counts = new_counts

    return {
        "final_answer": final, "strategy": strategy, "confidence": conf,
        "vote_counts": dict(vote_counts), "correct_votes": correct_votes,
        "total_votes": total, "vote_consistency": round(vote_consistency, 4),
        "wasted_votes": wasted, "refiner_used": refiner_used,
        "refiner_correct": refiner_correct,
    }

print("Voting logic ready")

In [ ]:
# CELL 10 -- Single question test

item = test_data[0]
q = item["question"]; gt = item["answer"]

print("=" * 65)
print(f"Q  : {q[:120]}")
print(f"GT : {gt}")

print("\n[CoT] 3 sample votes:")
for i in range(3):
    raw = generate_cot(q); pred = extract_mcq_answer(raw)
    print(f"  Vote {i+1}: '{pred}'  | ...{raw[-60:]}")

print("\n[Baseline] 3 sample votes:")
for i in range(3):
    raw = generate_baseline(q); pred = extract_mcq_answer(raw)
    print(f"  Vote {i+1}: '{pred}'")

print("\nSingle test passed. Run Cell 11 for full evaluation.")

In [ ]:
# CELL 11 -- Full Dual Evaluation Loop (N=900)
# Mode A: cot      (1.5B × 5, 'Let's think step by step')
# Mode B: baseline (1.5B × 5, standard prompt)
# Checkpoints every 50 questions. Estimated: ~3-4 hours on T4.

print(f"Dual evaluation: {len(test_data)} CSQA questions")
print(f"Votes per question: {CONFIG['n_votes']} CoT + {CONFIG['n_votes']} baseline")
print("-" * 65)

cot_results = []; base_results = []; start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        cot_results  = [r for r in lines if r.get("mode") == "cot"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from {start_idx} "
          f"(CoT: {len(cot_results)}, Base: {len(base_results)})")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="CSQA CoT Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = item["answer"]

    # ── CoT condition ────────────────────────────────────
    try:
        cot_votes = [extract_mcq_answer(generate_cot(question))
                     for _ in range(CONFIG["n_votes"])]
        c_dec = vote_and_decide(cot_votes, question, gt_answer)
        cot_results.append({
            "mode": "cot", "idx": idx, "question": question,
            "gt_answer": gt_answer,
            "final_answer": c_dec["final_answer"],
            "correct": c_dec["final_answer"] == gt_answer,
            "strategy": c_dec["strategy"],
            "confidence": c_dec["confidence"],
            "correct_votes": c_dec["correct_votes"],
            "total_votes": c_dec["total_votes"],
            "vote_consistency": c_dec["vote_consistency"],
            "wasted_votes": c_dec["wasted_votes"],
            "refiner_used": c_dec["refiner_used"],
            "refiner_correct": c_dec["refiner_correct"],
            "vote_counts": c_dec["vote_counts"],
        })
    except RuntimeError as e:
        cot_results.append({
            "mode": "cot", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0, "correct_votes": 0,
            "total_votes": CONFIG["n_votes"], "vote_consistency": 0.0,
            "wasted_votes": CONFIG["n_votes"], "refiner_used": False,
            "refiner_correct": None, "vote_counts": {}, "error": str(e),
        })

    # ── Baseline condition ───────────────────────────────
    try:
        base_votes = [extract_mcq_answer(generate_baseline(question))
                      for _ in range(CONFIG["n_votes"])]
        b_dec = vote_and_decide(base_votes, question, gt_answer)
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer,
            "final_answer": b_dec["final_answer"],
            "correct": b_dec["final_answer"] == gt_answer,
            "strategy": b_dec["strategy"],
            "confidence": b_dec["confidence"],
            "correct_votes": b_dec["correct_votes"],
            "total_votes": b_dec["total_votes"],
            "vote_consistency": b_dec["vote_consistency"],
            "wasted_votes": b_dec["wasted_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0, "correct_votes": 0,
            "total_votes": CONFIG["n_votes"], "vote_consistency": 0.0,
            "wasted_votes": CONFIG["n_votes"], "refiner_used": False,
            "refiner_correct": None, "vote_counts": {}, "error": str(e),
        })

    # ── Checkpoint ──────────────────────────────────────
    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in cot_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:4d}] CoT: {c_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

# Final save
with open(CONFIG["results_file"], "w") as f:
    for r in cot_results + base_results:
        f.write(json.dumps(r) + "\n")

c_c = sum(r["correct"] for r in cot_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nDone.")
print(f"  CoT      : {c_c}/{len(cot_results)} = {c_c/len(cot_results)*100:.1f}%")
print(f"  Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"  CoT vs Baseline : {(c_c/len(cot_results) - b_c/len(base_results))*100:+.1f} pts")

In [ ]:
# CELL 12 -- ANGLE 1: COMPUTE EFFICIENCY
# Both CoT and Baseline = 7.5B (1.5B × 5). Same compute, different prompt.

compute = CONFIG["solver_params_B"] * CONFIG["n_votes"]  # 7.5B
c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
c_wasted = sum(r["wasted_votes"] for r in cot_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)
c_ref = sum(r["refiner_used"] for r in cot_results)
b_ref = sum(r["refiner_used"] for r in base_results)

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY")
print("=" * 65)
print(f"\n  {'Config':<30} | {'Compute':>7} | {'Accuracy':>9}")
print(f"  {'-'*30}-+-{'':-<7}-+-{'':-<9}")
print(f"  {'Baseline (no CoT)':<30} | {compute:>5.1f}B  | {b_acc:>8.1f}%")
print(f"  {'CoT (think step by step)':<30} | {compute:>5.1f}B  | {c_acc:>8.1f}%")
print(f"  {'Guided (known result)':<30} | {'10.5B':>7} | {'75.8%':>9}")
print(f"\n  CoT vs Baseline : {c_acc - b_acc:+.1f} pts (same compute)")
print(f"  Guided vs CoT   : {75.8 - c_acc:+.1f} pts (guided uses 3B more compute)")
print(f"  Wasted votes: CoT={c_wasted}  Baseline={b_wasted}")
print(f"  Refiner   : CoT={c_ref}  Baseline={b_ref}")

angle1 = {
    "dataset": "CommonsenseQA", "n": len(cot_results), "compute_B": compute,
    "cot_accuracy": round(c_acc, 2), "baseline_accuracy": round(b_acc, 2),
    "guided_accuracy_known": 75.8,
    "cot_vs_baseline": round(c_acc - b_acc, 2),
    "guided_vs_cot": round(75.8 - c_acc, 2),
    "cot_wasted": c_wasted, "baseline_wasted": b_wasted,
    "cot_refiner": c_ref, "baseline_refiner": b_ref,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"Saved -> {CONFIG['angle1_file']}")

In [ ]:
# CELL 13 -- ANGLE 2: VOTE CONSISTENCY

c_cons = [r["vote_consistency"] for r in cot_results]
b_cons = [r["vote_consistency"] for r in base_results]
c_mean = np.mean(c_cons); b_mean = np.mean(b_cons)
lift   = c_mean / max(b_mean, 1e-6)

cot_wins  = sum(1 for c, b in zip(c_cons, b_cons) if c > b)
base_wins = sum(1 for c, b in zip(c_cons, b_cons) if b > c)
tied      = sum(1 for c, b in zip(c_cons, b_cons) if c == b)

def bucket(scores):
    return {
        "all_wrong (0%)":  sum(1 for s in scores if s == 0.0),
        "low (1-39%)":     sum(1 for s in scores if 0.0 < s < 0.4),
        "medium (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high (80-100%)":  sum(1 for s in scores if s >= 0.8),
    }

c_dist = bucket(c_cons); b_dist = bucket(b_cons)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY")
print("=" * 65)
print(f"\n  CoT mean consistency     : {c_mean*100:.1f}%")
print(f"  Baseline mean consistency: {b_mean*100:.1f}%")
print(f"  Consistency lift         : {lift:.2f}x")
print(f"\n  CoT wins / Baseline wins / Tied: {cot_wins} / {base_wins} / {tied}")
print(f"\n  {'Bucket':<22} | {'CoT':>6} | {'Baseline':>8}")
for bkt in ["all_wrong (0%)", "low (1-39%)", "medium (40-79%)", "high (80-100%)"]:
    cv, bv = c_dist[bkt], b_dist[bkt]
    print(f"  {bkt:<22} | {cv:>6} | {bv:>8}")

# Option bias check
print("\n  Option distribution (CoT final answers):")
cot_opts = Counter(r["final_answer"] for r in cot_results)
for opt in ['A','B','C','D','E']:
    n = cot_opts.get(opt, 0)
    print(f"    {opt}: {n} ({n/len(cot_results)*100:.1f}%)")

angle2 = {
    "cot_mean_consistency": round(c_mean, 4),
    "baseline_mean_consistency": round(b_mean, 4),
    "consistency_lift": round(lift, 4),
    "cot_wins": cot_wins, "baseline_wins": base_wins, "tied": tied,
    "cot_distribution": c_dist, "baseline_distribution": b_dist,
    "cot_option_dist": dict(cot_opts),
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"Saved -> {CONFIG['angle2_file']}")

In [ ]:
# CELL 14 -- ANGLE 3: CONFIDENCE CALIBRATION

def calibration_report(results, label):
    buckets = [
        ("Very High (>=0.80)",    lambda c: c >= 0.80, 0.90),
        ("High (0.60-0.80)",      lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium (0.40-0.60)",    lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low (<0.40)",           lambda c: c < 0.40, 0.25),
    ]
    n_total = len(results); ece = 0.0; calib = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])
    print(f"\n  [{label}]")
    for name, cond, mid in buckets:
        sub = [r for r in results if cond(r["confidence"])]
        if not sub: continue
        acc = sum(r["correct"] for r in sub) / len(sub)
        gap = abs(acc - mid)
        ece += (len(sub) / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"    {name:<24}: n={len(sub):>4}  acc={acc*100:.1f}%  expected={mid*100:.0f}%  "
              f"gap={gap:.3f}  {flag}")
        calib.append({"bucket": name, "count": len(sub),
                      "accuracy": round(acc, 4), "expected": mid, "gap": round(gap, 4)})
    hc = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"    ECE={ece:.4f}  High-conf={len(hc)}  Acc@high={hc_acc:.1f}%  "
          f"False-conf={false_conf}")
    return ece, calib, false_conf

print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION")
print("=" * 65)
c_ece, c_calib, c_false = calibration_report(cot_results,  "CoT")
b_ece, b_calib, b_false = calibration_report(base_results, "Baseline")

improve = (b_ece - c_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE: CoT={c_ece:.4f}  Baseline={b_ece:.4f}  Improvement={improve:.1f}%")
print(f"  False conf: CoT={c_false}  Baseline={b_false}")

angle3 = {
    "cot_ece": round(c_ece, 4), "baseline_ece": round(b_ece, 4),
    "ece_improvement_pct": round(improve, 2),
    "cot_false_conf": c_false, "baseline_false_conf": b_false,
    "cot_calibration": c_calib, "baseline_calibration": b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"Saved -> {CONFIG['angle3_file']}")

In [ ]:
# CELL 15 -- Final 3-Way Comparison Summary
# Paste your Guided results from the original CSQA notebook.

GUIDED_ACC = 75.8   # from original guided notebook
GUIDED_ECE = 0.141  # from original guided notebook

c_acc = sum(r["correct"] for r in cot_results)  / len(cot_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
c_ece = json.load(open(CONFIG["angle3_file"]))["cot_ece"]
b_ece = json.load(open(CONFIG["angle3_file"]))["baseline_ece"]

print("=" * 72)
print("  COMMONSENSEQA -- 3-WAY COMPARISON (Paper Table)")
print("=" * 72)
print(f"  N={len(cot_results)}  Seed=42  Solver=Qwen2.5-1.5B")
print()
print(f"  {'Condition':<32} | {'Compute':>7} | {'Accuracy':>9} | {'ECE':>7} | {'vs Baseline':>12}")
print(f"  {'-'*32}-+-{'':-<7}-+-{'':-<9}-+-{'':-<7}-+-{'':-<12}")
print(f"  {'Baseline (no guide, no CoT)':<32} | {'7.5B':>7} | {b_acc:>8.1f}% | {b_ece:>7.4f} | {'—':>12}")
print(f"  {'CoT (think step by step)':<32} | {'7.5B':>7} | {c_acc:>8.1f}% | {c_ece:>7.4f} | {c_acc-b_acc:>+11.1f}%")
print(f"  {'Guided (fine-tuned 3B guide)':<32} | {'10.5B':>7} | {GUIDED_ACC:>8.1f}% | {GUIDED_ECE:>7.4f} | {GUIDED_ACC-b_acc:>+11.1f}%")
print()
print(f"  KEY: Does Guided beat CoT?")
print(f"  Guided vs CoT: {GUIDED_ACC - c_acc:+.1f} pts")
if GUIDED_ACC > c_acc + 2:
    print("  RESULT: Guided clearly outperforms CoT. Fine-tuned guide is justified.")
elif abs(GUIDED_ACC - c_acc) <= 2:
    print("  RESULT: Marginal (<2 pts). Report honestly with the compute difference.")
else:
    print("  RESULT: CoT matches/beats Guided. Honest finding — report it.")